# Day 2, part 2: Read QC and quantification with nf-core/rnaseq

**Course:** Computational Workflows for Biomedical Data (CWBD), University of Tübingen, 2026 \
**Date:** 2026-09-29 \
**Author:** Laurens Hahn

**What:** Run nf-core/rnaseq on the two downloaded runs, read the QC, then check them against the tutors' run of all 16 samples.

**Why:** Counts are only as good as the reads behind them. This QC decides which samples go into the differential expression on day 3.

## Why nf-core

The paper's methods name tools but no versions, parameters or genome build, so the analysis cannot be repeated as written. The course uses nf-core pipelines instead.

### How does this approach make the analysis reproducible?

> Given that the current approach utilized by the authors lacks reproducibility, we will explore an alternative method by leveraging nf-core pipelines for data analysis.
>
> Please explain, how we will achieve reproducibility for the course  with this approach.

- Pinned pipeline version: -r 3.27.0
- Containers: -profile docker runs every tool from a fixed container image with fixed tool versions
- Pinned inputs: The raw data comes by SRA accession through fetchngs and the reference is a fixed Ensembl release (116, GRCm39)
- Everything is logged: pipeline_info/ records the parameters used, all software versions and execution reports
- Community standards → same tools: nf-core pipelines are peer reviewed, CI tested on every change and documented
- Contrast with the paper: "HISAT2, HT-Seq and DESeq2" with no versions, parameters or genome build given

## Running nf-core/rnaseq

### Which pipeline gives a count table and read QC from FASTQ files?

> You have successfully downloaded 2 of the fastq files we will use in our study.
>
> What is the next step if we want to first have a count table and check the quality of our fastq files? What is the pipeline called to do so?

**nf-core/rnaseq**

Input: FASTQs\
Output: QC, trimming and alignment and produces gene-level count matrices plus a MultiQC report

### What does nf-core/rnaseq do, and which tools does it use?

> Analyze the 2 files using an nf-core pipeline.
>
> What does this pipeline do?
>
> Which are the main tools that will be used in the pipeline?

1. Raw read QC and preprocessing: FastQC (read quality), fq lint (checks FASTQ integrity), Salmon on a subsample (infers strandedness when set to auto) and Trim Galore/Cutadapt (adapter and quality trimming)
2. Alignment and quantification: STAR + Salmon by default, with alternatives STAR + RSEM, HISAT2, Bowtie2 + Salmon or pseudoalignment with Salmon/Kallisto → tximport then summarises transcripts to gene-level count and TPM matrices
3. Post-alignment processing: SAMtools (sort, index, stats), Picard MarkDuplicates, StringTie (transcript assembly) and BEDTools/bedGraphToBigWig (coverage tracks)
4. QC: RSeQC (read distribution, gene body coverage, strandedness), Qualimap, dupRadar (duplication vs expression level), Preseq (library complexity), featureCounts biotype QC (for example, the rRNA share) and DESeq2 PCA and sample clustering
5. MultiQC collects all of it into one report

### Samplesheet

> As all other nf-core pipelines, the chosen pipeline takes in a samplesheet as input.
>
> Use Python and pandas to create the samplesheet for your 2 samples. Feel free to make use of the table you created earlier today.
>
> Choose your sample names wisely, they must be the connection of the results to the metadata. If you can't find the sample in the metadata later, the analysis was useless.

One row per sample with both FASTQ files. The sample name carries group and run accession, so every result maps straight back to the metadata.

In [ ]:
# Needs the FASTQs from part 1, which were deleted after the run: this cell fails now
# Packages
import pandas as pd
from pathlib import Path
import seaborn as sns

# import raw data
raw_data = pd.read_excel("conditions_runs_oxy_project.xlsx")

# "x" columns → one column each for condition and genotype
cond_cols = ["condition: Sal", "Condition: Oxy"]
geno_cols = ["Genotype: SNI", "Genotype: Sham"]
df = raw_data[raw_data["RNA-seq"] == "x"].copy()
df["condition"] = df[cond_cols].notna().idxmax(axis=1).str.split(": ").str[1]
df["genotype"] = df[geno_cols].notna().idxmax(axis=1).str.split(": ").str[1]

# remove unnecessary columns and sort by genotype, condition, and run
df = df[["Run", "genotype", "condition"]].sort_values(["genotype", "condition", "Run"]).reset_index(drop=True)

# import data
fastq_dir = Path("results/fetchngs/fastq").resolve()
ids = pd.read_csv("ids.csv", header=None, names=["Run"])
sheet = ids.merge(df[["Run", "genotype", "condition"]], on="Run")

# sample name carries group + run accession → results map straight back to the metadata
sheet["sample"] = sheet["genotype"] + "_" + sheet["condition"] + "_" + sheet["Run"]
sheet["fastq_1"] = sheet["Run"].map(lambda r: str(next(fastq_dir.glob(f"*_{r}_1.fastq.gz"))))
sheet["fastq_2"] = sheet["Run"].map(lambda r: str(next(fastq_dir.glob(f"*_{r}_2.fastq.gz"))))
sheet["strandedness"] = "auto"
sheet[["sample", "fastq_1", "fastq_2", "strandedness"]].to_csv("samplesheet_rnaseq.csv", index=False)

### Pipeline command

> post here the command you used to run nf-core/rnaseq

In [ ]:
# Already run (2 h 29 min). Fails now: the FASTQs were deleted after the run.
# The first attempt downloaded the reference (Ensembl 116, GRCm39) with
#   --fasta https://ftp.ensembl.org/pub/release-116/fasta/mus_musculus/dna/Mus_musculus.GRCm39.dna.primary_assembly.fa.gz
#   --gtf https://ftp.ensembl.org/pub/release-116/gtf/mus_musculus/Mus_musculus.GRCm39.116.gtf.gz
# This rerun reuses that reference plus prebuilt indices (see "How did the pipeline perform?" for why)
!nextflow run nf-core/rnaseq -r 3.27.0 -profile docker -c server.config \
  --input samplesheet_rnaseq.csv --outdir results/rnaseq \
  --fasta results/reference/Mus_musculus.GRCm39.dna.primary_assembly.fa \
  --gtf results/reference/Mus_musculus.GRCm39.116.gtf \
  --hisat2_index results/reference/hisat2 \
  --salmon_index results/reference/salmon_index_nodecoy \
  --aligner hisat2 --pseudo_aligner salmon

### Which parameters did you set, and why?

> Explain all the parameters you set and why you set them in this way.

| Parameter  | Why?  |
|---|---|
| -r 3.27.0  | Pins the pipeline version (latest release), for reproducibility  |
| -profile docker  |  Every tool runs in a versioned container, with no localinstalls |
| -c server.config  | It caps CPU and RAM to match available resources; otherwise nf-core's default requests (up to 72 GB) get refused  |
| --input  |  the samplesheet above |
| --outdir  | Under results/, so that it´s git-ignored  |
| --fasta, --gtf  | Ensembl 116 GRCm39, the current mouse assembly  |
| --aligner hisat2  | STAR (the default) needs about 32 GB RAM for a mouse index + HISAT2 fits, and it's also the paper's aligner |
| --pseudo_aligner salmon  | Needed with HISAT2: in nf-core/rnaseq, the HISAT2 route produces no counts on its own. Salmon gives the gene count table and the HISAT2 BAMs feed the alignment QC  |
| strandedness: auto  | Salmon detects the library orientation |

## Results of the two sample run

### How did the pipeline perform?

> How did the pipeline perform?

- Final run: 2 h 29 min wall time, 91 tasks succeeded, 0 failed. It used all 12 CPUs, peaked at 14 GB RAM and added up to about 15 h of task time
- Two problems on the way:
  - SALMON_INDEX was killed for running out of memory (exit 137). The default gentrome index uses the whole genome as decoy and needs 16 to 20 GB → Fix: a transcriptome-only Salmon index, passed with `--salmon_index`
  - In the first run only SNI_Oxy was aligned with HISAT2 → Sham_Oxy went through trimming and Salmon but never reached HISAT2_ALIGN, even though the run reported success. Cause: the HISAT2 index built during the run is emitted as a queue channel (the other indices get `.first()`), so HISAT2_ALIGN fires only once → Fix: pass the built index with `--hisat2_index` and rerun both samples
- Mapping: HISAT2 aligned 85.5 % of SNI_Oxy (76.9 % of pairs uniquely) and 83.1 % of Sham_Oxy (78.8 % uniquely) + Salmon mapped 62.9 % (SNI_Oxy) and 46.0 % (Sham_Oxy)
- Strandedness: `auto` detected reverse stranded (ISR) in both samples → Salmon (94.7 / 92.7 %) and RSeQC (85.6 / 88.8 % antisense) agree, so the check passed

![HISAT2 alignment](figures/hisat2_pe_plot-pct.png)

*HISAT2 alignment per sample. Both samples align above 83 %, most pairs uniquely, and the unaligned share is similar in both.*

### Is the read quality good enough?

> Explain the quality control steps. Are you happy with the quality and why. If not, why not. \
> Please give additional information on :
> - ribosomal rRNA
> - Duplication
> - GC content

- Base quality is good throughout, and the error rate after alignment is 0.36 %
- Adapter content fails in the raw reads because the inserts are short: Salmon's mean fragment length is 172 to 183 bp, so 150 bp reads run into the adapter. Trim Galore cut adapters from 54 to 58 % of reads, and after trimming adapter content passes
- Per base sequence content fails over the first ~12 bases. That is the usual random hexamer priming bias of TruSeq libraries, not a problem.
- Gene body coverage is even (5'-3' bias 1.03), so there are no signs of RNA degradation.
- Nucleus accumbens identity confirmed: striatal medium spiny neuron markers are high in both samples (TPM, SNI / Sham: Ppp1r1b 281 / 335, Penk 265 / 361, Drd1 73 / 125, Drd2 22 / 37).
- The two samples correlate well: Pearson 0.956 on log2(TPM + 1).

Verdict: usable, with reservations. The reads are clean in both samples, but SNI_Oxy is a low-complexity library (high duplication, more short fragments), and Sham_Oxy carries a lot of intronic signal (53.6 % of aligned reads), which is why its Salmon mapping rate is low.

![FastQC status](figures/fastqc_raw-status-check-heatmap.png)

*FastQC pass, warn and fail for every check and read file. Checks that fail in all files are typical RNA-seq artefacts; a check failing in only one sample would point to a sample specific problem.*

#### Ribosomal RNA

- featureCounts biotypes (SNI_Oxy / Sham_Oxy): nuclear rRNA 0.011 / 0.009 %, mitochondrial rRNA (Mt_rRNA) 2.3 / 3.5 %, protein coding 91.1 / 89.8 %, lncRNA 3.5 / 4.1 %, misc_RNA 2.0 / 1.0 %.
- rRNA is very low, so the poly(A) selection of the TruSeq kit worked. nf-core doesn't run SortMeRNA by default, and it wasn't needed here.
- The most overrepresented sequences in FastQC are not rRNA. They map to 7SL RNA (Rn7s2, Rn7s6), the RNA part of the signal recognition particle. It is abundant, short and not polyadenylated, so some always gets through poly(A) selection.
- Watch out: Rn7s2 alone is about 50 % of all TPM in SNI_Oxy. TPM divides by transcript length, so a ~300 nt RNA with many reads blows up. In raw counts it's about 2 %. Use counts for DE, not TPM.
- Sham_Oxy has more mitochondrial rRNA (3.5 vs 2.3 %), in line with its higher mt-Rnr2 TPM. Still far below a level that would matter.

![Biotypes](figures/featurecounts_biotype_plot-pct.png)

*Share of assigned reads per gene biotype. Protein coding dominates in both samples, rRNA is too small to see and Mt_rRNA is a thin band, slightly larger in Sham_Oxy.*

#### Duplication

- FastQC, share of reads left after deduplication: SNI_Oxy 37 to 45 %, Sham_Oxy 69 to 75 %.
- Picard: 69.8 % duplicates in SNI_Oxy vs 23.9 % in Sham_Oxy. The estimated library size is 5.4 M unique fragments for SNI_Oxy against 31.0 M for Sham_Oxy, with a similar number of aligned pairs (17.1 M vs 17.8 M).
- In RNA-seq, high duplication is partly expected: highly expressed genes produce identical fragments by chance. dupRadar separates the two effects. In a good library, duplication is near 0 for weakly expressed genes and only rises for strong ones. Sham_Oxy behaves like that (about 10 to 12 % at low expression). SNI_Oxy already sits at 35 to 46 % for weakly expressed genes, so even rare transcripts are heavily duplicated. That means PCR duplicates from low input material, not biology.
- RSeQC counts only 27.8 % of SNI_Oxy reads as non-duplicate proper pairs, vs 73.2 % for Sham_Oxy. It's the same effect, because RSeQC leaves the marked duplicates out.
- SNI_Oxy also lost the most pairs as too short (3.6 % vs 0.6 %) and has the most overrepresented sequences (about 6 % vs 0.3 %). All of this points to a low-input, low-complexity library.
- Consequence: SNI_Oxy's 20 M reads carry less information than the number suggests. Duplicates are not removed for DE analysis (standard for RNA-seq), but the sample is worth flagging.

![dupRadar](figures/dupradar.png)

*Duplication rate against gene expression. A good library starts near 0 % for weakly expressed genes and only rises for strong ones (Sham_Oxy). A curve that already starts high means PCR duplicates (SNI_Oxy).*

#### GC content

- Mean GC: SNI_Oxy 48 %, Sham_Oxy 47 %. That matches the mouse transcriptome (about 48 to 50 %).
- The shoulder is stronger in SNI_Oxy: 15 % of its reads fall between 58 and 64 % GC, vs 9 % in Sham_Oxy. That fits its higher duplication and larger share of overrepresented short RNAs.
- The main peak is where it should be. FastQC warns because the curve has a shoulder at about 60 to 62 % and a small extra peak at about 82 %. FastQC compares against a normal distribution, which RNA-seq never matches exactly, because a few very highly expressed transcripts (here 7SL RNA and mitochondrial transcripts) each add their own GC value in large numbers.
- No sign of contamination with another organism (that would give a clearly separate broad peak), and both samples show the same shape.
- Salmon's GC bias correction is off by default in nf-core. For the full analysis it would be worth turning on (`--extra_salmon_quant_args '--gcBias --seqBias'`).

![GC content](figures/fastqc_trimmed_per_sequence_gc_content_plot_Percentages.png)

*GC distribution of the trimmed reads. The main peak sits at 46 to 48 % in both samples, and the shoulder around 60 % is larger in SNI_Oxy.*

### What are the possible steps that could lead to poorer results?

> What are the possible steps that could lead to poorer results?

Wet lab
- Low RNA input or degraded RNA → PCR duplicates and low complexity (as seen in SNI_Oxy). The punches are small, and 2 animals are pooled per replicate.
- Poor poly(A) selection → more rRNA and intronic reads. Intronic reads are 36.7 % in SNI_Oxy, which is high for mRNA-seq but common in brain tissue.
- Short inserts → adapter read-through.
- Batch effects: the Oxy runs are about 10 % shallower than the Sal runs (6.94 vs 7.73 Gb, p = 0.004), so depth is confounded with treatment.

Processing
- Wrong strandedness setting. GEO names the TruSeq RNA kit, which is unstranded, but the data is clearly reverse stranded. Anyone who trusted the metadata and set `unstranded` would lose about half of the counts or count the wrong strand.
- A decoy-free Salmon index: genomic reads (intronic, intergenic) can be assigned to similar transcripts by mistake. That could be part of the low Salmon rate in Sham_Oxy.
- Reference and annotation versions. The paper used mm10 and Ensembl v90, we used GRCm39 and Ensembl 116. Gene IDs and models differ, so results can't be compared 1:1.
- Samples silently dropped by the pipeline (the HISAT2 bug here) or processes cut by resource limits.

Design
- n = 1 per group with only 2 of 16 samples: no statistics possible. The two samples also differ in pain state and both are Oxy, so any difference could be pain, library quality or animal.

![Genomic origin](figures/qualimap_genomic_origin-pct.png)

*Where the aligned reads fall: exonic, intronic or intergenic. Sham_Oxy has more intronic than exonic reads, and those are what the transcriptome-only Salmon index cannot map.*

### Would you exclude any samples? If yes, which and why?

> Would you exclude any samples? If yes, which and why?

Not at this stage, but both are flagged.

- **SNI_Oxy_SRR23195516**: good mapping (85.5 % HISAT2), but high technical duplication (69.8 % Picard, only 5.4 M unique fragments, dupRadar 35 to 46 % already at low expression) and more short fragments. This is the weaker library. Keep it, but watch whether it's an outlier in PCA once all 16 samples are in.
- **Sham_Oxy_SRR23195511**: good mapping (83.1 % HISAT2), low duplication (23.9 %), correct strandedness. The low Salmon rate (46.0 %) comes from a high intronic share (53.6 %), not from bad reads. It is a composition difference to keep in mind, not a reason to exclude.

![Picard duplicates](figures/picard_deduplication-pct.png)

*Unique vs duplicate aligned reads per sample (Picard). Sham_Oxy is mostly unique, SNI_Oxy mostly duplicates, the main reason it is the weaker sample.*

## Scaling up to all 16 samples

### What would you do next, what are the scientists trying to find out, and which packages would you use?

> What would you now do to continue the experiment? What are the scientists trying to figure out? Which packages on R or python would you use?

- Scale up: run fetchngs and rnaseq on all 16 runs, with -resume so the reference and index are reused
- The scientific question: how chronic neuropathic pain changes the transcriptional response of the reward circuit to oxycodone withdrawal and which regulators drive that (the paper's answer was HDAC1/2)
- Analysis steps:
  - DE with DESeq2 on the 2 × 2 design, including the interaction term. nf-core/differentialabundance runs this on the rnaseq count matrix
  - GO and pathway enrichment
  - Upstream regulator analysis
  - RRHO to compare the withdrawal signatures of SNI and Sham mice
- R packages: DESeq2 (or edgeR/limma-voom), tximport, clusterProfiler, RRHO2, pheatmap/ggplot2, EnhancedVolcano
- Python equivalents: PyDESeq2, decoupler (pathways and regulators), gseapy (enrichment), scanpy-style plotting with seaborn

The tutors did the scale up: nf-core/rnaseq on all 16 NAc samples with STAR and Salmon on GRCm38 (Ensembl, iGenomes), output in `rnaseq_out_allsamples/`. The differential expression follows on day 3.

### What does the MultiQC report of all 16 samples show?

Samples in the tutors' run are named by SRX. The cell maps them to runs and groups through ENA: the GEO title letters H, C, F and B are SNI_Oxy, Sham_Oxy, SNI_Sal and Sham_Sal, which matches the condition sheet for all 16. The table collects the key MultiQC numbers per sample.

In [1]:
import pandas as pd

mq = "rnaseq_out_allsamples/multiqc/star_salmon/multiqc_report_data/"

# The tutors' results are named by SRX. ENA gives SRX → SRR → GEO title, and the title letter
# encodes the group (checked against conditions_runs_oxy_project.xlsx, all 16 match)
ena = pd.read_csv("https://www.ebi.ac.uk/ena/portal/api/filereport?accession=PRJNA926667"
                  "&result=read_run&fields=run_accession,experiment_accession,sample_title&format=tsv", sep="\t")
letter = {"H": ("SNI", "Oxy"), "C": ("Sham", "Oxy"), "F": ("SNI", "Sal"), "B": ("Sham", "Sal")}
samples = pd.DataFrame({
    "sample": ena.experiment_accession,
    "run": ena.run_accession,
    "title": ena.sample_title,
    "surgery": ena.sample_title.str[0].map(lambda k: letter[k][0]),
    "treatment": ena.sample_title.str[0].map(lambda k: letter[k][1]),
})
samples["group"] = samples.surgery + "_" + samples.treatment
samples = samples.sort_values(["group", "title"]).set_index("sample")
samples.to_csv("samples_allsamples.csv")

stats = pd.read_csv(mq + "multiqc_general_stats.txt", sep="\t", index_col=0)
star = pd.read_csv(mq + "multiqc_star.txt", sep="\t", index_col=0)
biotype = pd.read_csv(mq + "multiqc_featurecounts_biotype_plot.txt", sep="\t", index_col=0)
biotype = biotype.div(biotype.sum(axis=1), axis=0) * 100

s = samples.index
qc = samples[["title", "run", "group"]].copy()
qc["M_pairs"] = stats.loc[s, "trim_galore-tg_total_reads"] / 2
qc["STAR_uniq_%"] = stats.loc[s, "star-uniquely_mapped_percent"]
qc["too_short_%"] = star.loc[s, "unmapped_tooshort_percent"]
qc["Picard_dup_%"] = stats.loc[s, "picard_mark_duplicates-PERCENT_DUPLICATION"]
qc["GC_%"] = stats.loc[s, "fastqc_raw-percent_gc"]
qc["rRNA_%"] = biotype.loc[s, "rRNA"]
qc["Mt_rRNA_%"] = biotype.loc[s, "Mt_rRNA"]
qc.round(1)

,title,run,group,M_pairs,STAR_uniq_%,too_short_%,Picard_dup_%,GC_%,rRNA_%,Mt_rRNA_%
sample,,,,,,,,,,
SRX19144482,H1,SRR23195517,SNI_Oxy,22.9,84.1,9.7,44.2,47.5,0.1,5.1
SRX19144486,H2,SRR23195516,SNI_Oxy,20.7,80.9,8.5,71.8,48.0,0.0,2.3
SRX19144490,H3,SRR23195509,SNI_Oxy,23.3,83.3,8.1,26.4,47.0,0.0,0.6
SRX19144494,H4,SRR23195508,SNI_Oxy,23.1,75.8,12.2,32.1,49.0,0.0,5.4
SRX19144481,F1,SRR23195518,SNI_Sal,26.4,68.4,16.8,81.4,56.0,24.4,17.5
SRX19144485,F2,SRR23195513,SNI_Sal,27.0,69.3,25.7,23.1,51.0,0.2,4.2
SRX19144489,F3,SRR23195510,SNI_Sal,24.6,55.4,34.6,88.0,56.0,21.3,3.2
SRX19144493,F4,SRR23195505,SNI_Sal,23.1,83.5,7.9,73.7,48.0,0.0,4.5
SRX19144480,C1,SRR23195519,Sham_Oxy,23.3,76.4,15.7,58.8,49.5,0.1,1.7


- **Depth is even:** 20.7 to 27.2 M read pairs after trimming, no sample is too shallow.
- **11 samples look normal:** 75 to 85 % uniquely mapped, GC 47 to 51.5 %.
- **5 samples stand out, with two different problems:**
  - **C4 (SRR23195506), F3 (SRR23195510), B4 (SRR23195507):** low unique mapping (26.1, 55.4, 54.5 %), many reads unmapped as "too short" (71.4, 34.6, 40.0 %) and raised GC (63, 56, 55 %). The GC rich overrepresented sequences match rDNA.
  - **F1 (SRR23195518), C2 (SRR23195514):** many reads on annotated rRNA genes (24.4 and 9.3 % of featureCounts), F1 also 17.5 % Mt_rRNA and 81.4 % duplication. Poor rRNA removal.
- **The other overrepresented sequences are 7SL RNA**, as in the two sample run, which is endogenous.

![STAR alignment](figures/allsamples_star_alignment_pct.png)

*STAR alignment per sample in the tutors' run. C4 is mostly "unmapped: too short" and F3 and B4 also have a large share of it. These are the human rRNA reads (next cell).*

### Is the GC rich signal human or mouse rRNA?

Each read is classified by 25-mers unique to the human (U13369) or mouse (BK000964) rDNA repeat, on the first 200k R1 reads per sample streamed from ENA. Slow (a few minutes), so not rerun.

In [ ]:
import gzip
import urllib.request

# The GC-rich overrepresented sequences match human rDNA (U13369) but not mouse rDNA (BK000964)
# Classify the first 200k R1 reads of every sample by k-mers unique to either rDNA repeat
K = 25
comp = str.maketrans("ACGT", "TGCA")

def ena_fasta(acc):
    txt = urllib.request.urlopen(f"https://www.ebi.ac.uk/ena/browser/api/fasta/{acc}").read().decode()
    return "".join(txt.splitlines()[1:]).upper()

def kmers(seq):
    rc = seq.translate(comp)[::-1]
    return {x[i:i + K] for x in (seq, rc) for i in range(len(x) - K + 1)}

mouse, human = kmers(ena_fasta("BK000964")), kmers(ena_fasta("U13369"))
mouse_only, human_only = mouse - human, human - mouse

def classify(run, n_reads=200_000):
    url = pd.read_csv(f"https://www.ebi.ac.uk/ena/portal/api/filereport?accession={run}"
                      "&result=read_run&fields=fastq_ftp&format=tsv", sep="\t").fastq_ftp[0].split(";")[0]
    n = m = h = 0
    with gzip.open(urllib.request.urlopen("https://" + url), "rt") as fq:
        for i, line in enumerate(fq):
            if i % 4 != 1:
                continue
            read = line.strip()
            ks = {read[j:j + K] for j in range(0, len(read) - K + 1, 8)}
            a, b = len(ks & mouse_only), len(ks & human_only)
            m += a >= 2 and a > b
            h += b >= 2 and b > a
            n += 1
            if n == n_reads:
                break
    return 100 * m / n, 100 * h / n

qc[["mouse_rDNA_%", "human_rDNA_%"]] = [classify(r) for r in qc.run]
qc.round(1)

,title,run,group,M_pairs,STAR_uniq_%,too_short_%,Picard_dup_%,GC_%,rRNA_%,Mt_rRNA_%,mouse_rDNA_%,human_rDNA_%
sample,,,,,,,,,,,,
SRX19144482,H1,SRR23195517,SNI_Oxy,22.9,84.1,9.7,44.2,47.5,0.1,5.1,3.4,5.0
SRX19144486,H2,SRR23195516,SNI_Oxy,20.7,80.9,8.5,71.8,48.0,0.0,2.3,2.5,2.0
SRX19144490,H3,SRR23195509,SNI_Oxy,23.3,83.3,8.1,26.4,47.0,0.0,0.6,1.3,1.0
SRX19144494,H4,SRR23195508,SNI_Oxy,23.1,75.8,12.2,32.1,49.0,0.0,5.4,2.8,2.3
SRX19144481,F1,SRR23195518,SNI_Sal,26.4,68.4,16.8,81.4,56.0,24.4,17.5,31.1,4.1
SRX19144485,F2,SRR23195513,SNI_Sal,27.0,69.3,25.7,23.1,51.0,0.2,4.2,4.4,16.8
SRX19144489,F3,SRR23195510,SNI_Sal,24.6,55.4,34.6,88.0,56.0,21.3,3.2,5.5,29.8
SRX19144493,F4,SRR23195505,SNI_Sal,23.1,83.5,7.9,73.7,48.0,0.0,4.5,2.1,1.2
SRX19144480,C1,SRR23195519,Sham_Oxy,23.3,76.4,15.7,58.8,49.5,0.1,1.7,1.8,9.9


- **C4, F3, B4 carry human rRNA:** 58.4, 29.8 and 24.1 % of reads match human rDNA only. STAR cannot place them on the mouse genome, so they end up as "too short".
- **F1 and C2 carry mouse rRNA:** 31.1 and 22.1 % mouse rDNA reads. These are mouse reads of the wrong kind, which cost depth and drive duplication up.
- **The human reads are a gradient, not yes or no:** F2 (SRR23195513) 16.8 %, B1 (SRR23195520) 10.6 %, C1 (SRR23195519) 9.9 %, while B2 (SRR23195515), B3 (SRR23195512), H3 (SRR23195509) and F4 (SRR23195505) are below 1.5 %.

The k-mer check only looks at 200k R1 reads per sample. A dedicated screen (FastQ Screen or BBSplit against mouse and human) would be the proper test.

**For day 3:** C4, F3 and B4 are out, because foreign reads can map onto mouse genes and add signal that is not biology. F1 and C2 stay but are flagged: mouse rRNA only costs depth, and dropping them too would leave two replicates in SNI_Sal and Sham_Oxy. That leaves 4 SNI_Oxy, 3 SNI_Sal, 3 Sham_Oxy and 3 Sham_Sal.

### How do the two samples from the two sample run compare with the tutors' run of the same libraries?

H2 is SNI_Oxy_SRR23195516 and C3 is Sham_Oxy_SRR23195511 from the two sample run. The two sample run used HISAT2 on GRCm39, the tutors' STAR on GRCm38, so only metrics that do not depend on the aligner are compared.

In [2]:
# Same libraries, two runs: MultiQC of the two sample run (hisat2) next to the tutors' (star_salmon)
def mq_metrics(mq_dir, samples):
    gs = pd.read_csv(mq_dir + "multiqc_general_stats.txt", sep="\t", index_col=0).loc[samples]
    bt = pd.read_csv(mq_dir + "multiqc_featurecounts_biotype_plot.txt", sep="\t", index_col=0).loc[samples]
    bt = bt.div(bt.sum(axis=1), axis=0) * 100
    aligned = gs[[f"qualimap_rnaseq-reads_aligned_{k}" for k in ("exonic", "intronic", "intergenic")]].sum(axis=1)
    return pd.DataFrame({
        "M_pairs": gs["trim_galore-tg_total_reads"] / 2,
        "adapter_%": gs["trim_galore-tg_pct_with_adapter"],
        "GC_%": gs["fastqc_raw-percent_gc"],
        "Picard_dup_%": gs["picard_mark_duplicates-PERCENT_DUPLICATION"],
        "intronic_%": 100 * gs["qualimap_rnaseq-reads_aligned_intronic"] / aligned,
        "rRNA_%": bt["rRNA"],
        "Mt_rRNA_%": bt["Mt_rRNA"],
    }, index=samples)

ours = mq_metrics("results/rnaseq/multiqc/hisat2/multiqc_report_data/", ["SNI_Oxy_SRR23195516", "Sham_Oxy_SRR23195511"])
ours.index = ["H2", "C3"]
tutors = mq_metrics(mq, qc.index[qc.title.isin(["H2", "C3"])].tolist())
tutors.index = qc.loc[tutors.index, "title"]
pd.concat({"two sample run (HISAT2)": ours, "tutors' 16 sample run (STAR)": tutors.loc[["H2", "C3"]]}).swaplevel().sort_index().round(1)

M_pairs  adapter_%  GC_%  Picard_dup_%  \
C3 tutors' 16 sample run (STAR)     21.5       57.1  47.0          25.4   
   two sample run (HISAT2)          21.5       57.1  47.0          23.9   
H2 tutors' 16 sample run (STAR)     20.7       54.0  48.0          71.8   
   two sample run (HISAT2)          20.7       54.0  48.0          69.8   

                                 intronic_%  rRNA_%  Mt_rRNA_%  
C3 tutors' 16 sample run (STAR)        53.0     0.0        3.5  
   two sample run (HISAT2)             53.6     0.0        3.5  
H2 tutors' 16 sample run (STAR)        37.5     0.0        2.3  
   two sample run (HISAT2)             36.7     0.0        2.3

Same libraries, same picture. Read pairs, adapter share, GC and the rRNA shares are identical, because they are measured before alignment or barely depend on it (nuclear rRNA is about 0.01 % in both). Duplication is slightly higher in the tutors' run (H2 71.8 vs 69.8 %, C3 25.4 vs 23.9 %) and the intronic share differs by less than 1 point; both come from STAR and HISAT2 placing reads a little differently.

So the reading from the two sample run holds: H2's high duplication is a property of the library itself, not something the HISAT2 alignment or the Picard step of the two sample run introduced, and C3's large intronic share is real, because the tutors' STAR run finds it too.